In [ ]:
import pandas as pd
import numpy as np
from src.features.process_race_header import preprocess_pipeline_race_header

/Users/axelcebille/Documents/GreyhoundRacingPrediction/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 0. Load race header dataset and trap bias statistics

In [2]:
full_race_header = pd.read_csv("../data/intermediate/07_all_race_header.csv")
trap_bias_stats = pd.read_csv("../data/processed/trap_bias_stats.csv")

In [3]:
full_race_header.head()

,raceTime,raceDate,raceId,raceTitle,raceNumber,raceType,raceHandicap,raceClass,raceDistance,racePrizes,raceGoing,raceForecast,raceTricast,meeting_Id,source_file
0,13:44:00,14/04/2018,1,NaN,13,Flat,False,A7,480.0,1st £92 | Others £35 | Race Total £267,10.0,(5-1) £26.94,(5-1-2) £60.36,337356,1.csv
1,11:07:00,14/04/2018,10,NaN,3,Flat,False,A3,480.0,1st £100 | Others £37 | Race Total £285,10.0,(1-4) £12.07,(1-4-6) £39.29,337356,10.csv
2,14:06:00,17/04/2019,100,NaN,1,Flat,False,A10,480.0,1st £105 | 2nd £40 | Others £40 | Race Total £305,-10.0,(1-2) £24.55,(1-2-3) £52.24,350084,100.csv
3,17:08:00,26/04/2019,1000,NaN,10,Flat,False,A8,380.0,1st £110 | 2nd £45 | Others £40 | Race Total £315,10.0,"(2-4) £23.87, (2-4) £23.87","(2-4-1) £36.97, (2-4-5) £34.51",350316,1000.csv
4,21:56:00,19/02/2014,10000,NaN,10,Flat,False,A1,400.0,1st £135 | Others £35 | Race Total £310,NaN,(3-1) £13.71,(3-1-5) £48.83,284014,10000.csv


# 1. Apply preliminary cleaning and processing

**TODO: add comments and def.**

In [4]:
race_header_cleaned = preprocess_pipeline_race_header(full_race_header)

In [5]:
race_header_cleaned.head()

,raceTime,raceDate,raceId,raceTitle,raceNumber,raceHandicap,raceClass,raceDistance,racePrizes,raceGoing,...,meeting_Id,source_file,raceType_Flat,raceType_Hurdles,raceYear,raceMonth,raceDayOfWeek,raceDayOfYear,classGroup,qualityLevel
0,13:44:00,2018-04-14,1,NaN,13,False,A7,480.0,1st £92 | Others £35 | Race Total £267,10.0,...,337356,1.csv,1,0,2018,4,5,104,graded_a,7
1,11:07:00,2018-04-14,10,NaN,3,False,A3,480.0,1st £100 | Others £37 | Race Total £285,10.0,...,337356,10.csv,1,0,2018,4,5,104,graded_a,11
2,14:06:00,2019-04-17,100,NaN,1,False,A10,480.0,1st £105 | 2nd £40 | Others £40 | Race Total £305,-10.0,...,350084,100.csv,1,0,2019,4,2,107,graded_a,4
3,17:08:00,2019-04-26,1000,NaN,10,False,A8,380.0,1st £110 | 2nd £45 | Others £40 | Race Total £315,10.0,...,350316,1000.csv,1,0,2019,4,4,116,graded_a,6
4,21:56:00,2014-02-19,10000,NaN,10,False,A1,400.0,1st £135 | Others £35 | Race Total £310,NaN,...,284014,10000.csv,1,0,2014,2,2,50,graded_a,13


# 2. Go fetch track names

An issue with the dataset is that the track names are missing, which is an important information to have, mainly to associate the trap bias.

So we have to go fetch it on the website where we got the data in the first place.

In [ ]:
import requests
# ---------------- API ----------------

def fetch_track_name(meeting_id: str):
    url_test = f"https://api.gbgb.org.uk/api/results/meeting/{meeting_id}?meeting={meeting_id}"
    response = requests.get(url=url_test, timeout=15)
    response.raise_for_status()
    r = response.json()
    response.close()

    if not r or not isinstance(r, list) or len(r) == 0:
        raise ValueError("Empty response for meeting_id")

    track_name = r[0].get("trackName")
    return track_name

In [ ]:
meeting_col = "meeting_Id"
meeting_ids = race_header_cleaned[meeting_col].astype(str).unique().tolist()
meeting_id_to_track = {}

for meeting_id in meeting_ids:
    if meeting_id in meeting_id_to_track:
        continue
    try:
        meeting_id_to_track[meeting_id] = fetch_track_name(meeting_id)
    except Exception:
        meeting_id_to_track[meeting_id] = None

race_header_cleaned["trackName"] = race_header_cleaned[meeting_col].astype(str).map(meeting_id_to_track)

# 3. Trap bias per track/distance 

In [ ]:
trap_lookup = (
    trap_bias_stats
    .set_index(["trackName", "raceDistance", "trapNumber"])["trapBiasResultPosition"]
    .to_dict()
)

for t in range(1, 7):
    race_header_cleaned[f"trap{t}Bias"] = race_header_cleaned.apply(
        lambda row, t=t: trap_lookup.get((row["trackName"], row["raceDistance"], t), np.nan),
        axis=1
    )

In [ ]:
race_header_cleaned

,raceTime,raceDate,raceId,raceTitle,raceNumber,raceHandicap,raceClass,raceDistance,racePrizes,raceGoing,...,raceDayOfYear,classGroup,qualityLevel,trackName,trap1Bias,trap2Bias,trap3Bias,trap4Bias,trap5Bias,trap6Bias
0,13:44:00,2018-04-14,1,NaN,13,False,A7,480.0,1st £92 | Others £35 | Race Total £267,10.0,...,104,graded_a,7,Perry Barr,3.0,3.0,3.0,3.0,4.0,3.0
1,13:44:00,2018-04-14,1,NaN,13,False,A7,480.0,1st £92 | Others £35 | Race Total £267,10.0,...,104,graded_a,7,Perry Barr,3.0,3.0,3.0,3.0,4.0,3.0
2,13:44:00,2018-04-14,1,NaN,13,False,A7,480.0,1st £92 | Others £35 | Race Total £267,10.0,...,104,graded_a,7,Perry Barr,3.0,3.0,3.0,3.0,4.0,3.0
3,13:44:00,2018-04-14,1,NaN,13,False,A7,480.0,1st £92 | Others £35 | Race Total £267,10.0,...,104,graded_a,7,Perry Barr,3.0,3.0,3.0,3.0,4.0,3.0
4,13:44:00,2018-04-14,1,NaN,13,False,A7,480.0,1st £92 | Others £35 | Race Total £267,10.0,...,104,graded_a,7,Perry Barr,3.0,3.0,3.0,3.0,4.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3253844,21:16:00,2023-12-27,999983,Christmas Owners' Bonus Series 22nd Trophy,12,False,A2,491.0,1st £215 | Others £65 Race Total £540,-10.0,...,361,graded_a,12,Central Park,3.0,3.0,3.0,3.0,4.0,3.0
3253845,21:16:00,2023-12-27,999983,Christmas Owners' Bonus Series 22nd Trophy,12,False,A2,491.0,1st £215 | Others £65 Race Total £540,-10.0,...,361,graded_a,12,Central Park,3.0,3.0,3.0,3.0,4.0,3.0
3253846,21:16:00,2023-12-27,999983,Christmas Owners' Bonus Series 22nd Trophy,12,False,A2,491.0,1st £215 | Others £65 Race Total £540,-10.0,...,361,graded_a,12,Central Park,3.0,3.0,3.0,3.0,4.0,3.0
3253847,21:16:00,2023-12-27,999983,Christmas Owners' Bonus Series 22nd Trophy,12,False,A2,491.0,1st £215 | Others £65 Race Total £540,-10.0,...,361,graded_a,12,Central Park,3.0,3.0,3.0,3.0,4.0,3.0


# 4. Final cleaning

## 4.1 Keep the usefull columns and fill by mean

In [ ]:
cols_to_remove = ["raceTime", "raceDate", "raceTitle", "raceNumber", "raceHandicap", 
                  "raceClass", "racePrizes", "raceGoing", "raceForecast", "raceTricast", 
                  "raceType_Flat", "raceType_Hurdles"]
cols_to_keep = [col for col in race_header_cleaned.columns if col not in cols_to_remove]

In [ ]:
final_race_header =race_header_cleaned[cols_to_keep].copy()
mean_val = final_race_header.mean(skipna=True)    # mean without outliers
final_race_header.fillna(mean_val, inplace=True)

In [ ]:
final_race_header.describe()

## 4.2 Keep only A graded races and all the races

In [ ]:
Aclass_race_header = final_race_header[final_race_header["classGroup"] == "graded_a"].copy()

In [ ]:
final_race_header.to_csv("../data/processed/all_race_header.csv", index=False)
Aclass_race_header.to_csv("../data/processed/graded_a_race_header.csv", index=False)